# 08 · Raw Decibel (dB) Calculation
Compute RMS raw decibel levels for all 6,558 clips in `data/processed/labels.csv`, save `data/processed/audio_rawdb.npy`, and verify.

In [1]:
# Cell 1 – interpreter setup & imports
import sys
from pathlib import Path

print('sys.executable:', sys.executable)
REPO_ROOT = Path.cwd().parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
print('REPO_ROOT     :', REPO_ROOT)

from src.preprocessing.audio import load_fixed
from src.preprocessing.intensity import raw_db
print('Successfully imported load_fixed from src.preprocessing.audio and raw_db from src.preprocessing.intensity')

sys.executable: C:\Users\LENOVO\miniconda3\envs\citypulse\python.exe
REPO_ROOT     : C:\Users\LENOVO\Desktop\CityPulse


Successfully imported load_fixed from src.preprocessing.audio and raw_db from src.preprocessing.intensity


In [2]:
# Cell 2 – read labels.csv and verify file hashes
import hashlib
import pandas as pd

labels_path = REPO_ROOT / 'data/processed/labels.csv'
features_path = REPO_ROOT / 'data/processed/audio_features.npy'

labels = pd.read_csv(labels_path)
print(f'labels.csv rows: {len(labels):,}, columns: {labels.columns.tolist()}')

def get_file_info(p):
    size = p.stat().st_size
    h = hashlib.sha256()
    with open(p, 'rb') as f:
        while chunk := f.read(65536):
            h.update(chunk)
    return size, h.hexdigest()

labels_size, labels_sha = get_file_info(labels_path)
feat_size, feat_sha = get_file_info(features_path)

print(f'labels.csv         size: {labels_size:,} bytes | SHA256: {labels_sha}')
print(f'audio_features.npy size: {feat_size:,} bytes | SHA256: {feat_sha}')

EXPECTED_LABELS_SHA = '8b4194ff7b2f47048c0e70f5ddfcc9c19d1994079405584dc8e86b9180d5f854'
EXPECTED_FEAT_SHA   = 'ed16a16f2a9e5c90aa61e925b461e0bce053c4654f1cc22a0812264630b1c73a'

assert labels_sha == EXPECTED_LABELS_SHA, f'labels.csv SHA mismatch: expected {EXPECTED_LABELS_SHA}, got {labels_sha}'
assert feat_sha == EXPECTED_FEAT_SHA, f'audio_features.npy SHA mismatch: expected {EXPECTED_FEAT_SHA}, got {feat_sha}'
print('Cell 2 SHA256 asserts passed ✓')

labels.csv rows: 6,558, columns: ['path', 'source', 'fold', 'original_class', 'app_class']
labels.csv         size: 535,784 bytes | SHA256: 8b4194ff7b2f47048c0e70f5ddfcc9c19d1994079405584dc8e86b9180d5f854
audio_features.npy size: 3,357,824 bytes | SHA256: ed16a16f2a9e5c90aa61e925b461e0bce053c4654f1cc22a0812264630b1c73a
Cell 2 SHA256 asserts passed ✓


In [3]:
# Cell 3 – compute raw_db for every row in order; keep waveforms for Cell 6
import time
import numpy as np
from tqdm import tqdm

n_clips = len(labels)
db_list = []
waveforms = []
failures = []

t0 = time.perf_counter()
for idx, row in tqdm(labels.iterrows(), total=n_clips, desc='Computing raw dB', mininterval=5.0):
    rel_path = row['path']
    abs_path = REPO_ROOT / rel_path
    try:
        y = load_fixed(abs_path)
        val = raw_db(y)
        db_list.append(val)
        waveforms.append(y)
    except Exception as e:
        failures.append((idx, rel_path, str(e)))

elapsed = time.perf_counter() - t0
print(f'Total computation time: {elapsed:.2f} s ({elapsed/60:.2f} min)')
print(f'Number of failures    : {len(failures)}')
if len(failures) == 0:
    db = np.array(db_list, dtype=np.float64)

Computing raw dB:   0%|          | 0/6558 [00:00<?, ?it/s]

Computing raw dB:   0%|          | 1/6558 [00:06<11:33:33,  6.35s/it]

Computing raw dB:   9%|▉         | 580/6558 [00:11<01:37, 61.34it/s] 

Computing raw dB:  17%|█▋        | 1120/6558 [00:16<01:06, 81.41it/s]

Computing raw dB:  28%|██▊       | 1813/6558 [00:21<00:46, 103.14it/s]

Computing raw dB:  36%|███▌      | 2370/6558 [00:26<00:39, 106.03it/s]

... (4 outputs left out)

Computing raw dB:  81%|████████  | 5319/6558 [00:51<00:10, 118.25it/s]

Computing raw dB:  91%|█████████ | 5949/6558 [00:56<00:05, 120.59it/s]

Computing raw dB: 100%|██████████| 6558/6558 [01:00<00:00, 107.85it/s]

Total computation time: 60.83 s (1.01 min)
Number of failures    : 0


In [4]:
# Cell 4 – assert shape, dtype, no NaN/inf, save to audio_rawdb.npy
assert len(failures) == 0, f'Encountered {len(failures)} failures during processing'
assert db.shape == (6558,), f'Expected db.shape == (6558,), got {db.shape}'
assert db.dtype == np.float64, f'Expected dtype float64, got {db.dtype}'
assert not np.isnan(db).any(), 'NaN values found in db'
assert not np.isinf(db).any(), 'Inf values found in db'

out_path = REPO_ROOT / 'data/processed/audio_rawdb.npy'
np.save(out_path, db)
print(f'Saved audio_rawdb.npy to: {out_path}')

min_db = float(np.min(db))
max_db = float(np.max(db))
mean_db = float(np.mean(db))
silent_count = int(np.sum(db < -120.0))

print(f'db min                       : {min_db:.4f} dB')
print(f'db max                       : {max_db:.4f} dB')
print(f'db mean                      : {mean_db:.4f} dB')
print(f'Values below -120 dB (silent): {silent_count}')
print('Cell 4 asserts passed ✓')

Saved audio_rawdb.npy to: C:\Users\LENOVO\Desktop\CityPulse\data\processed\audio_rawdb.npy
db min                       : -74.8055 dB
db max                       : -4.1796 dB
db mean                      : -25.6995 dB
Values below -120 dB (silent): 0
Cell 4 asserts passed ✓


In [5]:
# Cell 5 – reload audio_rawdb.npy and summarize by app_class and source
reloaded_db = np.load(REPO_ROOT / 'data/processed/audio_rawdb.npy')
assert np.array_equal(reloaded_db, db), 'Reloaded audio_rawdb.npy does not equal in-memory db exactly'
print('Reloaded audio_rawdb.npy matches in-memory db exactly ✓')

labels_with_db = labels.copy()
labels_with_db['raw_db'] = reloaded_db

print('\n=== Per-app_class raw dB summary ===')
class_summary = labels_with_db.groupby('app_class')['raw_db'].agg(['mean', 'median', 'min', 'max'])
print(class_summary.round(4).to_string())

print('\n=== Per-source raw dB summary ===')
source_summary = labels_with_db.groupby('source')['raw_db'].agg(['mean', 'median', 'min', 'max'])
print(source_summary.round(4).to_string())

Reloaded audio_rawdb.npy matches in-memory db exactly ✓

=== Per-app_class raw dB summary ===
                      mean   median      min      max
app_class                                            
construction      -24.3823 -23.1101 -55.7593  -4.6928
hvac              -27.6014 -25.5242 -53.8122 -12.3179
music_loudspeaker -25.0848 -24.2014 -55.6648  -6.7075
siren             -28.5078 -26.7606 -74.8055  -4.1796
traffic_horn      -24.8587 -25.1428 -61.7273  -5.9405

=== Per-source raw dB summary ===
           mean   median      min     max
source                                   
esc50  -18.7577 -18.1565 -40.4476 -4.1796
us8k   -25.9179 -24.4557 -74.8055 -4.6928


In [6]:
# Cell 6 – recompute raw_db for 5 random rows from scratch and assert match
sample_5 = labels.sample(5, random_state=7)
print(f'Spot check 5 random rows (random_state=7): {sample_5.index.tolist()}\n')

for idx in sample_5.index:
    row = labels.iloc[idx]
    # from in-memory waveform
    y_mem = waveforms[idx]
    db_mem = raw_db(y_mem)
    
    # from scratch
    y_scratch = load_fixed(REPO_ROOT / row['path'])
    db_scratch = raw_db(y_scratch)
    
    saved_val = reloaded_db[idx]
    print(f'Row {idx:4d} | app_class: {row["app_class"]:18} | source: {row["source"]:5} | in-memory: {db_mem:.6f} | from-scratch: {db_scratch:.6f} | saved: {saved_val:.6f}')
    
    assert db_mem == saved_val, f'Row {idx} in-memory value {db_mem} != saved {saved_val}'
    assert db_scratch == saved_val, f'Row {idx} from-scratch value {db_scratch} != saved {saved_val}'

print('\nALL 5 SPOT CHECKS PASSED (equal to saved values) ✓')

Spot check 5 random rows (random_state=7): [896, 5779, 351, 1175, 1188]

Row  896 | app_class: construction       | source: us8k  | in-memory: -23.115559 | from-scratch: -23.115559 | saved: -23.115559
Row 5779 | app_class: construction       | source: us8k  | in-memory: -21.306289 | from-scratch: -21.306289 | saved: -21.306289
Row  351 | app_class: construction       | source: us8k  | in-memory: -29.529857 | from-scratch: -29.529857 | saved: -29.529857
Row 1175 | app_class: music_loudspeaker  | source: us8k  | in-memory: -20.431979 | from-scratch: -20.431979 | saved: -20.431979
Row 1188 | app_class: music_loudspeaker  | source: us8k  | in-memory: -17.367245 | from-scratch: -17.367245 | saved: -17.367245

ALL 5 SPOT CHECKS PASSED (equal to saved values) ✓
